# Let's build the GPT Tokenizer（完整课堂版）

对照 Karpathy [minbpe](https://github.com/karpathy/minbpe) / Lecture 8。  
本 notebook **自包含**：helpers + `BasicTokenizer`（byte-level BPE）+ `RegexTokenizer`（GPT-2/4 切分 + 特殊 token）。

- 视频：https://www.youtube.com/watch?v=zduSFxRajkE
- 官方 Colab：https://colab.research.google.com/drive/1y0KnCFZvGVf_odSfcNAws6kcDD7HsI0L
- 笔记：Vault `nanoGPT · Let's build the GPT Tokenizer`

阅读顺序建议：先跑通 Section A/B，再看 C（regex 版）。看不懂的地方直接问。

## A. 公共工具：`get_stats` / `merge`

In [2]:
def get_stats(ids, counts=None):
    """统计相邻 pair 出现次数。ids: list[int]"""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """把 ids 里所有相邻 pair 替换成新 id。"""
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


# 快速自测
assert get_stats([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert merge([1, 2, 3, 1, 2], (1, 2), 4) == [4, 3, 4]
print('helpers ok')

helpers ok


## B. `BasicTokenizer`：byte-level BPE

要点：

1. 文本先 `encode('utf-8')` 变成 bytes（0..255）
2. 词表从 256 个 byte 起步
3. 训练：合并 `vocab_size - 256` 次
4. encode：按 merges **学到的先后顺序**（merge index 更小的优先）反复合并
5. decode：id → bytes 拼接 → UTF-8 解码

In [3]:
class BasicTokenizer:
    """最小 byte-level BPE（对齐 minbpe BasicTokenizer）。"""

    def __init__(self):
        self.merges = {}  # (int, int) -> int
        self.vocab = {idx: bytes([idx]) for idx in range(256)}  # int -> bytes

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)  # 0..255

        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = merge(ids, pair, idx)
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) count={stats[pair]}")

        self.merges = merges
        self.vocab = vocab

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            # 优先应用“更早学到”的 merge（id 更小）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids


print('BasicTokenizer defined')

BasicTokenizer defined


In [4]:
# 用一段稍长的文本训练并往返验证
train_text = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?
""".strip()

tok = BasicTokenizer()
tok.train(train_text, vocab_size=300, verbose=True)  # 256 + 44 merges

s = "hear me speak"
ids = tok.encode(s)
print('text:', repr(s))
print('ids:', ids)
print('decoded:', repr(tok.decode(ids)))
print('roundtrip:', tok.decode(tok.encode(train_text)) == train_text)
print('compression on train_text:', len(train_text.encode()), 'bytes ->', len(tok.encode(train_text)), 'tokens')
print('num merges:', len(tok.merges), 'vocab size:', len(tok.vocab))

merge 1/44: (101, 32) -> 256 (b'e ') count=5
merge 2/44: (101, 97) -> 257 (b'ea') count=4
merge 3/44: (58, 10) -> 258 (b':\n') count=3
merge 4/44: (116, 104) -> 259 (b'th') count=3
merge 5/44: (112, 257) -> 260 (b'pea') count=3
merge 6/44: (260, 107) -> 261 (b'peak') count=3
merge 7/44: (70, 105) -> 262 (b'Fi') count=2
merge 8/44: (262, 114) -> 263 (b'Fir') count=2
merge 9/44: (263, 115) -> 264 (b'Firs') count=2
merge 10/44: (264, 116) -> 265 (b'First') count=2
merge 11/44: (265, 32) -> 266 (b'First ') count=2
merge 12/44: (266, 67) -> 267 (b'First C') count=2
merge 13/44: (267, 105) -> 268 (b'First Ci') count=2
merge 14/44: (268, 116) -> 269 (b'First Cit') count=2
merge 15/44: (269, 105) -> 270 (b'First Citi') count=2
merge 16/44: (270, 122) -> 271 (b'First Citiz') count=2
merge 17/44: (271, 101) -> 272 (b'First Citize') count=2
merge 18/44: (272, 110) -> 273 (b'First Citizen') count=2
merge 19/44: (273, 258) -> 274 (b'First Citizen:\n') count=2
merge 20/44: (114, 256) -> 275 (b're ')

## C. `RegexTokenizer`：先正则切分，再在块内做 BPE

GPT-2/4 不会把整篇文档当成一条超长 byte 串硬 merge，而是：

1. 用正则把文本切成许多 chunk（词、数字、空白等）
2. **每个 chunk 内部**各自做 BPE（跨 chunk 不合并）
3. 可选：注册特殊 token（如 `<|endoftext|>`）

依赖：`regex`（支持 `\p{L}` 等 Unicode 属性）。本仓库用 uv：

```bash
uv add regex
```

In [5]:
# 本仓库已用 `uv add regex` 装好；若换环境再执行: uv add regex
import regex as re

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

print('sample GPT4 chunks:')
print(re.findall(GPT4_SPLIT_PATTERN, "Hello, world! It's 2024.\nNext"))

sample GPT4 chunks:
['Hello', ',', ' world', '!', ' It', "'s", ' ', '202', '4', '.\n', 'Next']


In [6]:
class RegexTokenizer:
    """对齐 minbpe RegexTokenizer：regex 预切分 + byte-level BPE + special tokens。"""

    def __init__(self, pattern=None):
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.special_tokens = {}
        self.inverse_special_tokens = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        text_chunks = re.findall(self.compiled_pattern, text)
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk_ids in ids:
                get_stats(chunk_ids, stats)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) count={stats[pair]}")

        self.merges = merges
        self.vocab = vocab

    def register_special_tokens(self, special_tokens):
        self.special_tokens = dict(special_tokens)
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def decode(self, ids):
        parts = []
        for idx in ids:
            if idx in self.vocab:
                parts.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                parts.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")
        return b"".join(parts).decode("utf-8", errors="replace")

    def _encode_chunk(self, text_bytes):
        ids = list(text_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def encode_ordinary(self, text):
        text_chunks = re.findall(self.compiled_pattern, text)
        ids = []
        for chunk in text_chunks:
            ids.extend(self._encode_chunk(chunk.encode("utf-8")))
        return ids

    def encode(self, text, allowed_special="none_raise"):
        if allowed_special == "all":
            special = self.special_tokens
        elif allowed_special == "none":
            special = {}
        elif allowed_special == "none_raise":
            special = {}
            assert all(token not in text for token in self.special_tokens)
        elif isinstance(allowed_special, set):
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")

        if not special:
            return self.encode_ordinary(text)

        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"
        special_chunks = re.split(special_pattern, text)
        ids = []
        for part in special_chunks:
            if part in special:
                ids.append(special[part])
            else:
                ids.extend(self.encode_ordinary(part))
        return ids


print('RegexTokenizer defined')

RegexTokenizer defined


In [7]:
rtok = RegexTokenizer()
rtok.train(train_text, vocab_size=300, verbose=False)
rtok.register_special_tokens({"<|endoftext|>": 300})

s = "Speak, speak.<|endoftext|>All:"
ids = rtok.encode(s, allowed_special="all")
print('text:', repr(s))
print('ids:', ids)
print('decoded:', repr(rtok.decode(ids)))

# 对比：同样词表预算下，Basic vs Regex 的切分差异
b_ids = tok.encode("Hello world!")
r_ids = rtok.encode("Hello world!", allowed_special="none")
print('Basic  Hello world! ->', b_ids)
print('Regex  Hello world! ->', r_ids)
print('Regex chunks:', re.findall(GPT4_SPLIT_PATTERN, "Hello world!"))

text: 'Speak, speak.<|endoftext|>All:'
ids: [83, 262, 44, 279, 46, 300, 65, 282, 58]
decoded: 'Speak, speak.<|endoftext|>All:'
Basic  Hello world! -> [72, 101, 287, 111, 32, 119, 111, 114, 108, 100, 33]
Regex  Hello world! -> [72, 101, 282, 111, 289, 111, 114, 108, 100, 33]
Regex chunks: ['Hello', ' world', '!']


## D. `decode`：比 encode 简单

```text
token ids → 查表变成 bytes → 拼起来 → UTF-8 解码成字符串
```

特殊 token：若 id 在 `inverse_special_tokens`，直接还原成原字符串（如 `<|endoftext|>`）。

`errors="replace"`：遇到残缺 UTF-8 字节时用替换字符顶上，避免直接崩溃。

## E. Tokenization 为什么会让模型“变怪”

下面这些不是模型数学错了，常常是 **切词方式** 导致的。

1. **空格敏感**：`" world"` 和 `"world"` 可能是不同 token
2. **数字被拆碎**：`123456` 可能切成多个数字块，算术更难
3. **同一词多种形态**：大小写、前导空格 → 不同 id
4. **跨语言不均**：英文常更短，其他语言可能更碎、更贵（占更多 context）
5. **特殊 token**：训练/对话格式依赖它们；乱用或泄漏会干扰模型

所以 Karpathy 说：很多 LLM 怪行为要先怀疑 tokenizer。

## F. 本课收束清单

你已经覆盖：

- BPE 训练：stats → merge → 重写序列 → merges/vocab
- Basic：byte-level 全文 BPE
- Regex：预切分 + 块内 BPE + 特殊 token
- encode：`encode` → `encode_ordinary` → `_encode_chunk`
- decode：id → bytes → 字符串

下一节对照 `tiktoken`（真实 GPT-2 / GPT-4 词表）。之后可回工程 nanoGPT 做 baseline 对照。

In [8]:
# 快速感受：前导空格会改变 tokenization
examples = ["world", " world", "World", "123456", "你好"]
print("RegexTokenizer examples:")
for s in examples:
    ids = rtok.encode(s, allowed_special="none")
    print(f"{s!r:10} -> {ids}  decoded={rtok.decode(ids)!r}")

print("\nchunks for 'Hello world!':", re.findall(GPT4_SPLIT_PATTERN, "Hello world!"))
print("chunks for '123456':", re.findall(GPT4_SPLIT_PATTERN, "123456"))
print("chunks for '你好世界':", re.findall(GPT4_SPLIT_PATTERN, "你好世界"))

RegexTokenizer examples:
'world'    -> [119, 111, 114, 108, 100]  decoded='world'
' world'   -> [289, 111, 114, 108, 100]  decoded=' world'
'World'    -> [87, 111, 114, 108, 100]  decoded='World'
'123456'   -> [49, 50, 51, 52, 53, 54]  decoded='123456'
'你好'       -> [228, 189, 160, 229, 165, 189]  decoded='你好'

chunks for 'Hello world!': ['Hello', ' world', '!']
chunks for '123456': ['123', '456']
chunks for '你好世界': ['你好世界']


## G. 对照 `tiktoken`（真实 GPT 词表）

`tiktoken` 是 OpenAI 官方 tokenizer。我们手写的是 **算法玩具**；这里看的是 **生产词表**。

| 编码名 | 大致对应 |
|--------|----------|
| `gpt2` | GPT-2 / 早期 GPT-3 |
| `cl100k_base` | GPT-4 / GPT-3.5-turbo 一类 |

依赖：本仓库已 `uv add tiktoken`。换环境再执行一次即可。

看什么：

1. `"world"` vs `" world"` — 不同 id（前导空格是独立子词）
2. `123456` — 常被拆成 `123` + `456`
3. 中文 — 往往更碎（更多 token / 更贵 context）
4. 同一句在 gpt2 vs cl100k 上 **id 不同、切法也可能不同**

In [9]:
# uv add tiktoken  （本仓库已装）
import tiktoken

gpt2 = tiktoken.get_encoding("gpt2")
cl100k = tiktoken.get_encoding("cl100k_base")  # GPT-4 family


def show(enc, name, text):
    ids = enc.encode(text)
    # 每个 id 对应的原始 bytes（比 decode 成字符串更直观）
    pieces = [enc.decode_single_token_bytes(i) for i in ids]
    print(f"[{name}] {text!r}")
    print(f"  n_tokens={len(ids)}  ids={ids}")
    print(f"  pieces={pieces}")


examples = ["world", " world", "World", "123456", "你好世界", "Hello world"]

print("=== gpt2 ===")
for s in examples:
    show(gpt2, "gpt2", s)

print("\n=== cl100k_base ===")
for s in examples:
    show(cl100k, "cl100k", s)

=== gpt2 ===
[gpt2] 'world'
  n_tokens=1  ids=[6894]
  pieces=[b'world']
[gpt2] ' world'
  n_tokens=1  ids=[995]
  pieces=[b' world']
[gpt2] 'World'
  n_tokens=1  ids=[10603]
  pieces=[b'World']
[gpt2] '123456'
  n_tokens=2  ids=[10163, 29228]
  pieces=[b'123', b'456']
[gpt2] '你好世界'
  n_tokens=8  ids=[19526, 254, 25001, 121, 10310, 244, 45911, 234]
  pieces=[b'\xe4\xbd', b'\xa0', b'\xe5\xa5', b'\xbd', b'\xe4\xb8', b'\x96', b'\xe7\x95', b'\x8c']
[gpt2] 'Hello world'
  n_tokens=2  ids=[15496, 995]
  pieces=[b'Hello', b' world']

=== cl100k_base ===
[cl100k] 'world'
  n_tokens=1  ids=[14957]
  pieces=[b'world']
[cl100k] ' world'
  n_tokens=1  ids=[1917]
  pieces=[b' world']
[cl100k] 'World'
  n_tokens=1  ids=[10343]
  pieces=[b'World']
[cl100k] '123456'
  n_tokens=2  ids=[4513, 10961]
  pieces=[b'123', b'456']
[cl100k] '你好世界'
  n_tokens=5  ids=[57668, 53901, 3574, 244, 98220]
  pieces=[b'\xe4\xbd\xa0', b'\xe5\xa5\xbd', b'\xe4\xb8', b'\x96', b'\xe7\x95\x8c']
[cl100k] 'Hello world'
  n_toke

In [ ]:
# 和我们玩具 RegexTokenizer 并排（rtok 需先跑过 Section C）
print("side-by-side on ' world' / '123456' / '你好世界':")
for s in [" world", "123456", "你好世界"]:
    toy = rtok.encode(s, allowed_special="none")
    g2 = gpt2.encode(s)
    c4 = cl100k.encode(s)
    print(f"{s!r}")
    print(f"  toy RegexTokenizer n={len(toy)}  {toy}")
    print(f"  gpt2               n={len(g2)}  {g2}")
    print(f"  cl100k             n={len(c4)}  {c4}")

print("\n结论提示：")
print("- id 数字不同是正常的（词表不同）")
print("- 看的是切法模式：空格敏感、数字拆碎、中文更碎")
print("- 玩具词表太小，中文几乎还是 byte；真实词表会 merge 出更多中文块")

side-by-side on ' world' / '123456' / '你好世界':
' world'
  toy RegexTokenizer n=5  [289, 111, 114, 108, 100]
  gpt2               n=1  [995]
  cl100k             n=1  [1917]
'123456'
  toy RegexTokenizer n=6  [49, 50, 51, 52, 53, 54]
  gpt2               n=2  [10163, 29228]
  cl100k             n=2  [4513, 10961]
'你好世界'
  toy RegexTokenizer n=12  [228, 189, 160, 229, 165, 189, 228, 184, 150, 231, 149, 140]
  gpt2               n=8  [19526, 254, 25001, 121, 10310, 244, 45911, 234]
  cl100k             n=5  [57668, 53901, 3574, 244, 98220]

结论提示：
- id 数字不同是正常的（词表不同）
- 看的是切法模式：空格敏感、数字拆碎、中文更碎
- 玩具词表太小，中文几乎还是 byte；真实词表会 merge 出更多中文块


### 快速读结果（跑完上一格后对照）

你本地跑出来应类似：

- `world` / ` world` / `World` → **三个不同 id**（都是 1 token，但是不同条目）
- `123456` → 常为 **2** tokens：`123` + `456`
- `你好世界`：gpt2 更碎（常 ~8），cl100k 稍好（常 ~5），仍远贵过英文同类长度
- `Hello world` → `Hello` + ` world`（空格粘在第二个词上）

这和 Section E 的“怪现象”一一对应——不是模型算错，是切词规则。

做完这节，Tokenizer 课正式收束。下一步建议回工程 nanoGPT baseline 对照实验。